In [1]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import math
from tqdm import tqdm
import os

In [2]:
rave_sim_dir = Path('/mnt/d/rave-sim-main/rave-sim-main')
simulations_dir = Path('/mnt/d/rave-sim-main/rave-sim-main/output')
scratch_dir = simulations_dir

In [3]:
sys.path.insert(0, str(rave_sim_dir / "big-wave"))
#print(str(rave_sim_dir / "big-wave"))
import multisim
import config
import util
import propagation

In [ ]:
times=np.arange(1,393)

for ii in times:
    config_dict = {
        "sim_params": {
            "is2d":'false',
            "N": 2**28,
            "dx":propagation.max_dx(0.01, 2e-7, 2**28, propagation.convert_energy_wavelength(10000)),
            "z_detector":1.7,
            "detector_size": 20e-3,
            "detector_pixel_size_x": 1e-4,
            "detector_pixel_size_y": 1,
            "chunk_size": 256 * 1024 * 1024 // 16,  # use 256MB chunks
        },
        "use_disk_vector": False,
        "save_final_u_vectors": False,
        "dtype": "c8",
        "multisource": {
            "type": "points",
            "energy_range": [8900-25, 8900+25],
            "x_range": [-10*1e-6/2.355, 10*1e-6/2.355],
            "z": 0.0,
            "nr_source_points": 1,
            "seed": 260208,
            #"spectrum": "/mnt/d/rave-sim-main/rave-sim-main/spectrum/spectrum_25keV.h5",
        },
        "elements": [
            {
                "type": "sample",
                "z_start": 0.02,
                "pixel_size_x": 2 * 1e-7,
                "pixel_size_z": 2 * 1e-7,
                "grid_path":"/mnt/d/rave-sim-main/rave-sim-main/grid/260208-full/shockwave_"+str(ii)+"_2e-4_2e-7_001.npy",
                "materials":[['SiO2',2.65],['C8H8',1.06]],
                "x_positions":[0*1e-4],
            },
        ],
    }
    file = open("/mnt/d/rave-sim-main/rave-sim-main/grid/260208-full/shockwave_"+str(ii)+"_001.txt", 'r')
    content=file.read()
    config_dict["elements"][0]["materials"] = eval(content)
    file.close()
    print("dx: ", config_dict["sim_params"]["dx"])
    print("N: ", config_dict["sim_params"]["N"])
    print("Species: ",len(config_dict["elements"][0]["materials"]))
    sim_path = multisim.setup_simulation(config_dict, Path("."), simulations_dir)
    computed = config.load(Path(sim_path / 'computed.yaml'))
    
    #print("cutoff angles:", computed['cutoff_angles'])
    #print("source points:", computed['source_points'])
    for i in tqdm(range(config_dict["multisource"]["nr_source_points"])):
        os.system(f"CUDA_VISIBLE_DEVICES=0 /mnt/d/rave-sim-main/rave-sim-main/fast-wave/build-Release/fastwave -s {i} {sim_path}")
    wavefronts = util.load_wavefronts_filtered(sim_path, x_range=(-10/2*1e-6, 10/2*1e-6))
    print("nr sources loaded:", len(wavefronts))
    wavef= [result[0] for result in wavefronts]
    wf = np.sum(wavef, axis=0)
    print("nr phase steps:", wf.shape[0])
    print("nr detector pixels:", wf.shape[1])
    sp = config_dict["sim_params"]
    detector_x = util.detector_x_vector(sp["detector_size"], sp["detector_pixel_size_x"])
    plt.plot(wf[0])
    from contextlib import redirect_stdout
    with open('/mnt/d/rave-sim-main/rave-sim-main/notebooks/shockwavetest_251225/results260209/D1720_260209_output_shocksample_shot_timestamp'+str(ii)+'.txt', 'w') as file:
        with redirect_stdout(file):
            for i in range(len(wf[0])):
                print(wf[0][i])
    # print(detector_x)

2026-02-09 08:59:12,102 INFO: Setting up simulation


dx:  8.294591680169107e-11
N:  268435456
Species:  2
2 2


2026-02-09 08:59:12,343 INFO: Finished setting up simulation in /mnt/d/rave-sim-main/rave-sim-main/output/2026/02/20260209_085912169172
  0%|                                                                                             | 0/1 [00:00<?, ?it/s]

[2026-02-09 08:59:12.571] [info] 2D mode:
[2026-02-09 08:59:12.571] [info] Running simulation /mnt/d/rave-sim-main/rave-sim-main/output/2026/02/20260209_085912169172/00000000
[2026-02-09 08:59:13.229] [info] Simulating optical element 1/1
[2026-02-09 09:00:42.810] [info] Elapsed time for optical element: 90201.99 ms


100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [01:30<00:00, 90.77s/it]
2026-02-09 09:00:43,165 INFO: Setting up simulation


...

[2026-02-09 19:04:45.722] [info] Simulation finished in 103.460119636 seconds
nr sources loaded: 1
nr phase steps: 1
nr detector pixels: 200
dx:  8.294591680169107e-11
N:  268435456
Species:  296
296 296


2026-02-09 19:04:47,390 INFO: Finished setting up simulation in /mnt/d/rave-sim-main/rave-sim-main/output/2026/02/20260209_190447054950
  0%|                                                                                             | 0/1 [00:00<?, ?it/s]

[2026-02-09 19:04:47.634] [info] 2D mode:
[2026-02-09 19:04:47.635] [info] Running simulation /mnt/d/rave-sim-main/rave-sim-main/output/2026/02/20260209_190447054950/00000000
[2026-02-09 19:04:48.284] [info] Simulating optical element 1/1
[2026-02-09 19:06:21.070] [info] Elapsed time for optical element: 92419.58 ms


100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [01:34<00:00, 94.02s/it]
2026-02-09 19:06:21,451 INFO: Setting up simulation


[2026-02-09 19:06:21.294] [info] Simulation finished in 102.24851175 seconds
nr sources loaded: 1
nr phase steps: 1
nr detector pixels: 200
dx:  8.294591680169107e-11
N:  268435456
Species:  281
281 281


In [ ]:
times=np.arange(1,393)

for ii in times:
    config_dict = {
        "sim_params": {
            "is2d":'false',
            "N": 2**28,
            "dx":propagation.max_dx(0.01, 2e-7, 2**28, propagation.convert_energy_wavelength(10000)),
            "z_detector":0.3,
            "detector_size": 18e-3,
            "detector_pixel_size_x": 1e-4,
            "detector_pixel_size_y": 1,
            "chunk_size": 256 * 1024 * 1024 // 16,  # use 256MB chunks
        },
        "use_disk_vector": False,
        "save_final_u_vectors": False,
        "dtype": "c8",
        "multisource": {
            "type": "points",
            "energy_range": [8900-25, 8900+25],
            "x_range": [-10*1e-6/2.355, 10*1e-6/2.355],
            "z": 0.0,
            "nr_source_points": 1,
            "seed": 260208,
            #"spectrum": "/mnt/d/rave-sim-main/rave-sim-main/spectrum/spectrum_25keV.h5",
        },
        "elements": [
            {
                "type": "sample",
                "z_start": 0.01,
                "pixel_size_x": 2 * 1e-7,
                "pixel_size_z": 2 * 1e-7,
                "grid_path":"/mnt/d/rave-sim-main/rave-sim-main/grid/260208-full/shockwave_"+str(ii)+"_2e-4_2e-7_001.npy",
                "materials":[['SiO2',2.65],['C8H8',1.06]],
                "x_positions":[0*1e-4],
            },
        ],
    }
    file = open("/mnt/d/rave-sim-main/rave-sim-main/grid/260208-full/shockwave_"+str(ii)+"_001.txt", 'r')
    content=file.read()
    config_dict["elements"][0]["materials"] = eval(content)
    file.close()
    print("dx: ", config_dict["sim_params"]["dx"])
    print("N: ", config_dict["sim_params"]["N"])
    print("Species: ",len(config_dict["elements"][0]["materials"]))
    sim_path = multisim.setup_simulation(config_dict, Path("."), simulations_dir)
    computed = config.load(Path(sim_path / 'computed.yaml'))
    
    #print("cutoff angles:", computed['cutoff_angles'])
    #print("source points:", computed['source_points'])
    for i in tqdm(range(config_dict["multisource"]["nr_source_points"])):
        os.system(f"CUDA_VISIBLE_DEVICES=0 /mnt/d/rave-sim-main/rave-sim-main/fast-wave/build-Release/fastwave -s {i} {sim_path}")
    wavefronts = util.load_wavefronts_filtered(sim_path, x_range=(-10/2*1e-6, 10/2*1e-6))
    print("nr sources loaded:", len(wavefronts))
    wavef= [result[0] for result in wavefronts]
    wf = np.sum(wavef, axis=0)
    print("nr phase steps:", wf.shape[0])
    print("nr detector pixels:", wf.shape[1])
    sp = config_dict["sim_params"]
    detector_x = util.detector_x_vector(sp["detector_size"], sp["detector_pixel_size_x"])
    plt.plot(wf[0])
    from contextlib import redirect_stdout
    with open('/mnt/d/rave-sim-main/rave-sim-main/notebooks/shockwavetest_251225/results260208/260208_output_shocksample_shot_timestamp'+str(ii)+'.txt', 'w') as file:
        with redirect_stdout(file):
            for i in range(len(wf[0])):
                print(wf[0][i])
    # print(detector_x)

2026-02-08 18:25:15,642 INFO: Setting up simulation


dx:  8.294591680169107e-11
N:  268435456
Species:  2
2 2


2026-02-08 18:25:14,046 INFO: Finished setting up simulation in /mnt/d/rave-sim-main/rave-sim-main/output/2026/02/20260208_182515718906
  0%|                                                                                             | 0/1 [00:00<?, ?it/s]

[2026-02-08 18:25:14.272] [info] 2D mode:
[2026-02-08 18:25:14.272] [info] Running simulation /mnt/d/rave-sim-main/rave-sim-main/output/2026/02/20260208_182515718906/00000000
[2026-02-08 18:25:14.918] [info] Simulating optical element 1/1
[2026-02-08 18:26:48.812] [info] Elapsed time for optical element: 93800.18 ms


100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [01:35<00:00, 95.08s/it]
2026-02-08 18:26:49,190 INFO: Setting up simulation


...

[2026-02-09 02:59:13.597] [info] 2D mode:
[2026-02-09 02:59:13.597] [info] Running simulation /mnt/d/rave-sim-main/rave-sim-main/output/2026/02/20260209_025913153358/00000000
[2026-02-09 02:59:14.176] [info] Simulating optical element 1/1
[2026-02-09 03:00:47.236] [info] Elapsed time for optical element: 93067.67 ms


100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [01:34<00:00, 94.19s/it]
2026-02-09 03:00:47,609 INFO: Setting up simulation


[2026-02-09 03:00:47.445] [info] Simulation finished in 95.72694019 seconds
nr sources loaded: 1
nr phase steps: 1
nr detector pixels: 179
dx:  8.294591680169107e-11
N:  268435456
Species:  266
266 266


2026-02-09 03:00:48,761 INFO: Finished setting up simulation in /mnt/d/rave-sim-main/rave-sim-main/output/2026/02/20260209_030048532242
  0%|                                                                                             | 0/1 [00:00<?, ?it/s]

[2026-02-09 03:00:48.980] [info] 2D mode:
[2026-02-09 03:00:48.980] [info] Running simulation /mnt/d/rave-sim-main/rave-sim-main/output/2026/02/20260209_030048532242/00000000
